# **Imports**

In [84]:
%pip install iterative-stratification


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [85]:
%pip install nlpaug transformers torch -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [86]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
import nlpaug.augmenter.word as naw
import random
import nltk
import re
from collections import Counter
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import torch.nn as nn
from sklearn.metrics import f1_score
import time


In [87]:
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('omw-1.4')

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Malak\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\Malak\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\Malak\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Malak\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

# **Dataset Preprocessing**

In [88]:
data = pd.read_csv("../train.csv")

In [89]:
print("Dataset Info:")
print(data.info())

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 159571 entries, 0 to 159570
Data columns (total 8 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   id             159571 non-null  object
 1   comment_text   159571 non-null  object
 2   toxic          159571 non-null  int64 
 3   severe_toxic   159571 non-null  int64 
 4   obscene        159571 non-null  int64 
 5   threat         159571 non-null  int64 
 6   insult         159571 non-null  int64 
 7   identity_hate  159571 non-null  int64 
dtypes: int64(6), object(2)
memory usage: 9.7+ MB
None


In [90]:
print("\nDataset Description:")
print(data.describe())


Dataset Description:
               toxic   severe_toxic        obscene         threat  \
count  159571.000000  159571.000000  159571.000000  159571.000000   
mean        0.095844       0.009996       0.052948       0.002996   
std         0.294379       0.099477       0.223931       0.054650   
min         0.000000       0.000000       0.000000       0.000000   
25%         0.000000       0.000000       0.000000       0.000000   
50%         0.000000       0.000000       0.000000       0.000000   
75%         0.000000       0.000000       0.000000       0.000000   
max         1.000000       1.000000       1.000000       1.000000   

              insult  identity_hate  
count  159571.000000  159571.000000  
mean        0.049364       0.008805  
std         0.216627       0.093420  
min         0.000000       0.000000  
25%         0.000000       0.000000  
50%         0.000000       0.000000  
75%         0.000000       0.000000  
max         1.000000       1.000000  


In [91]:
data.head()

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0


In [92]:
print("\nMissing Values:")
data.isnull().sum()


Missing Values:


id               0
comment_text     0
toxic            0
severe_toxic     0
obscene          0
threat           0
insult           0
identity_hate    0
dtype: int64

In [93]:
# Count the number of duplicate rows
duplicate_count = data.duplicated().sum()
print(f"Number of duplicate rows before removal: {duplicate_count}")

Number of duplicate rows before removal: 0


# **Data Spliting**

In [94]:



X = data[["comment_text"]]
y = data[["toxic","severe_toxic","obscene","threat","insult","identity_hate"]]

# =====================================================
# First Split: 70% Train / 30% Temp
# =====================================================

msss1 = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=42
)

train_idx, temp_idx = next(msss1.split(X, y))

train_df = data.iloc[train_idx].reset_index(drop=True)
temp_df  = data.iloc[temp_idx].reset_index(drop=True)

# =====================================================
# Second Split: 15% Validation / 15% Test
# =====================================================

X_temp = temp_df[["comment_text"]]

y_temp = temp_df[[
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate"
]]

msss2 = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=42
)

val_idx, test_idx = next(msss2.split(X_temp, y_temp))

val_df = temp_df.iloc[val_idx].reset_index(drop=True)
test_df = temp_df.iloc[test_idx].reset_index(drop=True)

# =====================================================
# Check Sizes
# =====================================================

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

total = len(data)

print(f"Train: {len(train_df)/total:.2%}")
print(f"Validation: {len(val_df)/total:.2%}")
print(f"Test: {len(test_df)/total:.2%}")

Train: (111699, 8)
Validation: (23936, 8)
Test: (23936, 8)
Train: 70.00%
Validation: 15.00%
Test: 15.00%


# **Data Augmentation**


In [95]:
print("Training samples:", len(train_df))

print("\nLabel counts:")
print(train_df[[
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate"
]].sum())

rare_rows = train_df[
    (train_df["severe_toxic"] == 1) |
    (train_df["threat"] == 1) |
    (train_df["identity_hate"] == 1)
]

print("\nUnique rows to augment:", len(rare_rows))

Training samples: 111699

Label counts:
toxic            10706
severe_toxic      1116
obscene           5914
threat             335
insult            5514
identity_hate      983
dtype: int64

Unique rows to augment: 2086


In [96]:
aug_sub = naw.SynonymAug(aug_src='wordnet', aug_p=0.15)
aug_swap = naw.RandomWordAug(action="swap", aug_p=0.1)

In [97]:
# current counts -> target counts (adjust to taste)
targets = {
    'threat': 8000,
    'identity_hate': 8000,
    'severe_toxic': 8000,
}

In [98]:
def augment_row(text, n_versions=1):
    outputs = []
    for i in range(n_versions):
        try:
            if i % 2 == 0:
                new_text = aug_sub.augment(text)
            else:
                new_text = aug_swap.augment(text)
            new_text = new_text[0] if isinstance(new_text, list) else new_text
            outputs.append(new_text)
        except Exception as e:
            print(f"Skipped a row due to: {e}")
    return outputs

In [99]:
augmented_rows = []

for label, target_count in targets.items():
    subset = train_df[train_df[label] == 1]
    current_count = len(subset)
    needed = target_count - current_count

    if needed <= 0:
        print(f"{label}: already at/above target, skipping")
        continue

    print(f"{label}: {current_count} -> {target_count} (generating {needed} new rows)")

    # how many augmented versions per original row (rounded up)
    per_row = max(1, needed // current_count + 1)

    count_generated = 0
    for _, row in subset.iterrows():
        if count_generated >= needed:
            break
        versions = augment_row(row['comment_text'], n_versions=per_row)
        for v in versions:
            if count_generated >= needed:
                break
            new_row = row.copy()
            new_row['comment_text'] = v
            augmented_rows.append(new_row)
            count_generated += 1

aug_df = pd.DataFrame(augmented_rows)
print(f"\nTotal augmented rows generated: {len(aug_df)}")

threat: 335 -> 8000 (generating 7665 new rows)
identity_hate: 983 -> 8000 (generating 7017 new rows)
severe_toxic: 1116 -> 8000 (generating 6884 new rows)

Total augmented rows generated: 21566


In [100]:
train_augmented = pd.concat([train_df, aug_df], ignore_index=True)
train_augmented = train_augmented.sample(frac=1, random_state=42).reset_index(drop=True)

print("New label counts:")

print(
    train_augmented[
        [
            "toxic",
            "severe_toxic",
            "obscene",
            "threat",
            "insult",
            "identity_hate"
        ]
    ].sum()
)


New label counts:
toxic            31314
severe_toxic     11446
obscene          22317
threat            8951
insult           21957
identity_hate    10824
dtype: int64


# **Tokenizer + Vocabulary**


In [101]:

label_cols = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

def simple_tokenize(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)          # remove URLs
    text = re.sub(r"[^a-z0-9\s']", " ", text)              # keep letters/numbers/apostrophes
    text = re.sub(r"\s+", " ", text).strip()
    return text.split()

# Build vocabulary from TRAINING data only (never fit on val/test)
def build_vocab(texts, min_freq=2, max_vocab_size=30000):
    counter = Counter()
    for text in texts:
        counter.update(simple_tokenize(text))

    vocab = {'<PAD>': 0, '<UNK>': 1}
    for word, freq in counter.most_common(max_vocab_size):
        if freq >= min_freq:
            vocab[word] = len(vocab)
    return vocab

vocab = build_vocab(train_augmented['comment_text'])
print(f"Vocab size: {len(vocab)}")

Vocab size: 30002


In [102]:
MAX_LEN = 150 # adjust based on your comment length distribution

def encode_text(text, vocab, max_len=MAX_LEN):
    tokens = simple_tokenize(text)
    ids = [vocab.get(tok, vocab['<UNK>']) for tok in tokens[:max_len]]
    if len(ids) == 0:
        ids = [vocab['<UNK>']]
    return ids

# quick check on comment length distribution to justify MAX_LEN
lengths = train_augmented['comment_text'].apply(lambda x: len(simple_tokenize(x)))
print(lengths.describe())
print(f"95th percentile length: {lengths.quantile(0.95)}")

count    133265.000000
mean         67.095261
std         110.944391
min           0.000000
25%          15.000000
50%          33.000000
75%          71.000000
max        1407.000000
Name: comment_text, dtype: float64
95th percentile length: 231.0


In [103]:
class ToxicDataset(Dataset):
    def __init__(self, df, vocab, max_len=MAX_LEN):
        self.texts = df['comment_text'].values
        self.labels = df[label_cols].values.astype('float32')
        self.vocab = vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        ids = encode_text(self.texts[idx], self.vocab, self.max_len)
        return torch.tensor(ids, dtype=torch.long), torch.tensor(self.labels[idx])

def collate_fn(batch):
    sequences, labels = zip(*batch)
    lengths = torch.tensor([len(seq) for seq in sequences])
    padded = pad_sequence(sequences, batch_first=True, padding_value=vocab['<PAD>'])
    labels = torch.stack(labels)
    return padded, labels, lengths

In [104]:
BATCH_SIZE = 64

# Build vocab from AUGMENTED TRAINING data only
vocab = build_vocab(train_augmented['comment_text'])
print(f"Vocab size: {len(vocab)}")

train_dataset = ToxicDataset(train_augmented, vocab, max_len=150)
val_dataset   = ToxicDataset(val_df, vocab, max_len=150)
test_dataset  = ToxicDataset(test_df, vocab, max_len=150)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

Vocab size: 30002
Train batches: 2083
Val batches: 374
Test batches: 374


# **Training Different RNN architectures**


In [114]:
class RNNClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128,
                 num_layers=2, bidirectional=True, dropout=0.3,
                 num_classes=6, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.rnn = nn.RNN(embed_dim, hidden_dim, num_layers=num_layers,
                           batch_first=True, bidirectional=bidirectional,
                           dropout=dropout if num_layers > 1 else 0.0)
        self.dropout = nn.Dropout(dropout)
        mult = 2 if bidirectional else 1
        self.fc = nn.Linear(hidden_dim * mult * 2, num_classes)  # *2 for mean+max

    def forward(self, x, lengths):
        embedded = self.embedding(x)
        lengths = lengths.clamp(min=1)
        packed = nn.utils.rnn.pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_out, _ = self.rnn(packed)
        outputs, _ = nn.utils.rnn.pad_packed_sequence(packed_out, batch_first=True)
        mask = (outputs.abs().sum(dim=2) != 0).unsqueeze(2).float()
        mean_pool = (outputs * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        max_pool = outputs.masked_fill(mask == 0, -1e9).max(dim=1)[0]
        combined = torch.cat([mean_pool, max_pool], dim=1)
        return self.fc(self.dropout(combined))

In [106]:
rnn_configs = {
    "Simple_RNN":            dict(num_layers=1, bidirectional=False, dropout=0.0),
    "Stacked_RNN":           dict(num_layers=2, bidirectional=False, dropout=0.0),
    "Deep_RNN":              dict(num_layers=3, bidirectional=False, dropout=0.0),
    "Bidirectional_RNN":     dict(num_layers=1, bidirectional=True,  dropout=0.0),
    "Stacked_BiRNN":         dict(num_layers=2, bidirectional=True,  dropout=0.0),
    "RNN_Dropout":           dict(num_layers=1, bidirectional=False, dropout=0.3),
    "Stacked_RNN_Dropout":   dict(num_layers=2, bidirectional=False, dropout=0.3),
}

In [107]:

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    for x, y, lengths in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x, lengths)
        loss = criterion(logits, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)  # helps vanilla RNN stability
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

@torch.no_grad()
def evaluate(model, loader, criterion, threshold=0.5):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []
    for x, y, lengths in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x, lengths)
        loss = criterion(logits, y)
        total_loss += loss.item()

        probs = torch.sigmoid(logits)
        preds = (probs >= threshold).float()
        all_preds.append(preds.cpu())
        all_labels.append(y.cpu())

    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()

    f1_macro = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    f1_micro = f1_score(all_labels, all_preds, average='micro', zero_division=0)

    return total_loss / len(loader), f1_macro, f1_micro, all_preds, all_labels

In [ ]:
def run_experiment(model_class, config_name, config, vocab_size,
                    train_loader, val_loader, pos_weight=None, epochs=8, lr=1e-3):
    print(f"\n{'='*50}\nTraining: {config_name}\n{'='*50}")

    model = model_class(
        vocab_size=vocab_size, embed_dim=128, hidden_dim=128,
        num_classes=6, **config
    ).to(device)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)

    history = {"train_loss": [], "val_loss": [], "val_f1_macro": [], "val_f1_micro": []}
    best_f1_macro = 0
    best_state = None

    for epoch in range(epochs):
        start = time.time()
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
        val_loss, f1_macro, f1_micro, _, _ = evaluate(model, val_loader, criterion)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_f1_macro"].append(f1_macro)
        history["val_f1_micro"].append(f1_micro)
        if f1_macro > best_f1_macro:
            best_f1_macro = f1_macro
            best_state = model.state_dict()
        elapsed = time.time() - start
        print(f"Epoch {epoch+1}/{epochs} | train_loss={train_loss:.4f} | "
              f"val_loss={val_loss:.4f} | val_F1_macro={f1_macro:.4f} | "
              f"val_F1_micro={f1_micro:.4f} | {elapsed:.1f}s")

    model.load_state_dict(best_state)
    return model, history, best_f1_macro

In [ ]:
vocab_size =  30002
rnn_results = {}
rnn_models = {}
pos_counts = train_augmented[label_cols].sum().values
neg_counts = len(train_augmented) - pos_counts
pos_weight = torch.tensor(np.clip(neg_counts / pos_counts, 1, 10), dtype=torch.float32).to(device)

rnn_results = {}
rnn_models = {}

for name, config in rnn_configs.items():
    model, history, best_f1 = run_experiment(
        RNNClassifier, name, config, vocab_size,
        train_loader, val_loader, pos_weight=pos_weight, epochs=4, lr=1e-3
    )
    rnn_results[name] = {"history": history, "best_val_f1_macro": best_f1}
    rnn_models[name] = model

best_rnn_name = max(rnn_results, key=lambda k: rnn_results[k]["best_val_f1_macro"])
print(f"\n🏆 Best RNN architecture: {best_rnn_name} (val F1 macro = {rnn_results[best_rnn_name]['best_val_f1_macro']:.4f})")


Training: Simple_RNN
Epoch 1/8 | train_loss=0.4999 | val_loss=0.2668 | val_F1_macro=0.3881 | val_F1_micro=0.4842 | 43.4s
Epoch 2/8 | train_loss=0.3001 | val_loss=0.2552 | val_F1_macro=0.4101 | val_F1_micro=0.5266 | 43.9s
Epoch 3/8 | train_loss=0.2232 | val_loss=0.2456 | val_F1_macro=0.4495 | val_F1_micro=0.5844 | 43.7s
Epoch 4/8 | train_loss=0.1816 | val_loss=0.2403 | val_F1_macro=0.4352 | val_F1_micro=0.5724 | 43.3s
Epoch 5/8 | train_loss=0.1595 | val_loss=0.2696 | val_F1_macro=0.4903 | val_F1_micro=0.6330 | 43.3s
Epoch 6/8 | train_loss=0.1409 | val_loss=0.2739 | val_F1_macro=0.4255 | val_F1_micro=0.5588 | 43.1s
Epoch 7/8 | train_loss=0.1229 | val_loss=0.2820 | val_F1_macro=0.4649 | val_F1_micro=0.6115 | 43.4s
Epoch 8/8 | train_loss=0.1165 | val_loss=0.3028 | val_F1_macro=0.4833 | val_F1_micro=0.6319 | 43.3s

Training: Stacked_RNN
Epoch 1/8 | train_loss=0.5233 | val_loss=0.2875 | val_F1_macro=0.3722 | val_F1_micro=0.4560 | 61.1s
Epoch 2/8 | train_loss=0.3265 | val_loss=0.2719 | val_F

In [112]:
import numpy as np
from sklearn.metrics import f1_score

best_thr, best_micro = 0.5, 0
for thr in np.arange(0.1, 0.95, 0.02):
    preds = (val_probs >= thr).astype(int)
    micro = f1_score(val_labels, preds, average='micro', zero_division=0)
    if micro > best_micro:
        best_micro, best_thr = micro, thr

print(f"Best global threshold: {best_thr:.2f}, micro F1: {best_micro:.4f}")

# also check weighted at this threshold
preds = (val_probs >= best_thr).astype(int)
print("weighted F1:", f1_score(val_labels, preds, average='weighted', zero_division=0))

Best global threshold: 0.64, micro F1: 0.6819
weighted F1: 0.6919035633244879


In [111]:
from sklearn.metrics import precision_recall_curve
import numpy as np

def tune_thresholds(model, val_loader):
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for x, y, lengths in val_loader:
            x = x.to(device)
            logits = model(x, lengths)
            all_probs.append(torch.sigmoid(logits).cpu())
            all_labels.append(y)
    all_probs = torch.cat(all_probs).numpy()
    all_labels = torch.cat(all_labels).numpy()

    best_thresholds = []
    for i, col in enumerate(label_cols):
        precision, recall, thresholds = precision_recall_curve(all_labels[:, i], all_probs[:, i])
        f1s = 2 * precision * recall / (precision + recall + 1e-9)
        best_idx = np.argmax(f1s[:-1])
        best_thresholds.append(thresholds[best_idx])
        print(f"{col}: threshold={thresholds[best_idx]:.3f}, f1={f1s[best_idx]:.3f}")
    return np.array(best_thresholds), all_probs, all_labels

best_thresholds, val_probs, val_labels = tune_thresholds(rnn_models[best_rnn_name], val_loader)
final_preds = (val_probs >= best_thresholds).astype(int)

from sklearn.metrics import classification_report
print(classification_report(val_labels, final_preds, target_names=label_cols, zero_division=0))

toxic: threshold=0.559, f1=0.752
severe_toxic: threshold=0.534, f1=0.432
obscene: threshold=0.770, f1=0.750
threat: threshold=0.890, f1=0.434
insult: threshold=0.509, f1=0.682
identity_hate: threshold=0.925, f1=0.306
               precision    recall  f1-score   support

        toxic       0.80      0.71      0.75      2294
 severe_toxic       0.33      0.62      0.43       240
      obscene       0.74      0.75      0.75      1267
       threat       0.41      0.46      0.43        71
       insult       0.64      0.73      0.68      1182
identity_hate       0.34      0.28      0.31       211

    micro avg       0.69      0.70      0.69      5265
    macro avg       0.54      0.59      0.56      5265
 weighted avg       0.70      0.70      0.70      5265
  samples avg       0.06      0.06      0.06      5265



In [ ]:
from sklearn.metrics import classification_report
_, _, _, preds, labels = evaluate(rnn_models['Bidirectional_RNN'], val_loader, nn.BCEWithLogitsLoss())
print(classification_report(labels, preds, target_names=label_cols, zero_division=0))

               precision    recall  f1-score   support

        toxic       0.88      0.63      0.73      2294
 severe_toxic       0.34      0.48      0.39       240
      obscene       0.81      0.66      0.73      1267
       threat       0.36      0.31      0.33        71
       insult       0.72      0.59      0.65      1182
identity_hate       0.30      0.29      0.30       211

    micro avg       0.75      0.60      0.67      5265
    macro avg       0.57      0.49      0.52      5265
 weighted avg       0.77      0.60      0.67      5265
  samples avg       0.05      0.05      0.05      5265



In [ ]:
from sklearn.metrics import precision_recall_curve

probs = torch.sigmoid(torch.tensor(logits))  # or however you stored val logits
best_thresholds = {}
for i, col in enumerate(label_cols):
    precision, recall, thresholds = precision_recall_curve(all_labels[:, i], all_probs[:, i])
    f1s = 2 * precision * recall / (precision + recall + 1e-9)
    best_idx = f1s[:-1].argmax()  # thresholds has len-1 vs precision/recall
    best_thresholds[col] = thresholds[best_idx]
    print(f"{col}: best_threshold={thresholds[best_idx]:.3f}, f1={f1s[best_idx]:.3f}")

NameError: name 'logits' is not defined